# Customer Analytics -- RFM Segmentation, CLTV Modeling & Behavioral Clustering

**Author:** Mehmet Isik | **Date:** 2026-03 | **Kernel:** Python 3.11

---

## 1. Introduction

Understanding guest behavior is the cornerstone of modern hospitality revenue management.
This notebook provides an end-to-end customer analytics pipeline applied to a hotel group
operating across multiple properties.

**Dataset at a glance:**

| Metric | Value |
|--------|-------|
| Unique customers | ~12 000 |
| Transactions | ~53 000 |
| Date range | 2024-01 to 2025-06 |
| Channels | OTA, Direct, Travel Agent, Corporate |

**What we cover:**

1. **RFM Analysis** -- Recency / Frequency / Monetary scoring and segment assignment
2. **CLTV Modeling** -- BG-NBD + Gamma-Gamma probabilistic lifetime value
3. **K-Means Clustering** -- Behavioral segmentation with radar profiling
4. **Business Recommendations** -- Actionable takeaways per segment

All visualizations use **Plotly** for Kaggle-friendly interactivity.

---
## 2. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datetime import timedelta

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# CLTV modeling
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

# Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries loaded successfully.')

---
## 3. Data Loading & Initial Exploration

In [ ]:
# ----- Adjust paths for Kaggle vs local -----
# On Kaggle, upload the CSVs as a dataset and use:
#   Local: DATA_DIR = '../data/synthetic/'
# Locally:
DATA_DIR = '/kaggle/input/hotel-intelligence-platform-data/'

transactions = pd.read_csv(f'{DATA_DIR}transactions.csv', parse_dates=['transaction_date'])
customers    = pd.read_csv(f'{DATA_DIR}customer_profiles.csv')

print(f'Transactions : {transactions.shape[0]:,} rows x {transactions.shape[1]} cols')
print(f'Customers    : {customers.shape[0]:,} rows x {customers.shape[1]} cols')
transactions.head(3)

In [ ]:
customers.head(3)

In [ ]:
# Quick sanity checks
print('Transaction date range :', transactions['transaction_date'].min().date(),
      'to', transactions['transaction_date'].max().date())
print('Unique customers (txn) :', transactions['customer_id'].nunique())
print('Missing values (txn)   :')
print(transactions.isnull().sum()[transactions.isnull().sum() > 0])
print()
transactions.describe()

---
## 4. RFM Analysis

RFM segments customers along three dimensions:

| Metric | Meaning | Good direction |
|--------|---------|----------------|
| **Recency** | Days since last transaction | Lower is better |
| **Frequency** | Number of transactions | Higher is better |
| **Monetary** | Total revenue generated | Higher is better |

In [ ]:
# Reference date = one day after last transaction
reference_date = transactions['transaction_date'].max() + timedelta(days=1)
print('Reference date:', reference_date.date())

rfm = (
    transactions
    .groupby('customer_id')
    .agg(
        recency   = ('transaction_date', lambda x: (reference_date - x.max()).days),
        frequency = ('transaction_id', 'nunique'),
        monetary  = ('total_revenue', 'sum')
    )
    .reset_index()
)

print(f'RFM table: {rfm.shape[0]:,} customers')
rfm.describe()

### 4.1 Quintile Scoring (1-5)

Each metric is binned into quintiles. For **Recency**, rank 5 means
the *most recent* (best); for Frequency & Monetary, rank 5 means the highest.

In [ ]:
# Recency: lower days = better = higher score
rfm['R'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency & Monetary: higher = better = higher score
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Total'] = rfm['R'] + rfm['F'] + rfm['M']

rfm.head(10)

### 4.2 Segment Assignment

We map RFM scores to business-meaningful labels using a rule-based approach.

In [ ]:
def assign_segment(row):
    """Rule-based RFM segment assignment."""
    r, f, m = row['R'], row['F'], row['M']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and m >= 3:
        return 'Potential'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Lost'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

seg_counts = rfm['segment'].value_counts()
print('Segment distribution:')
print(seg_counts)
print(f'\nTotal customers: {seg_counts.sum():,}')

### 4.3 Segment Distribution (Pie Chart)

In [ ]:
color_map = {
    'Champion':    '#2ecc71',
    'Loyal':       '#3498db',
    'New Customer':'#9b59b6',
    'Potential':   '#f1c40f',
    'At Risk':     '#e67e22',
    'Lost':        '#e74c3c'
}

fig = px.pie(
    values=seg_counts.values,
    names=seg_counts.index,
    title='RFM Segment Distribution',
    color=seg_counts.index,
    color_discrete_map=color_map,
    hole=0.35
)
fig.update_traces(textinfo='percent+label', pull=[0.05]*len(seg_counts))
fig.update_layout(width=700, height=500)
fig.show()

### 4.4 RFM Scatter Plot (Recency vs Monetary)

In [ ]:
fig = px.scatter(
    rfm,
    x='recency',
    y='monetary',
    size='frequency',
    color='segment',
    color_discrete_map=color_map,
    hover_data=['customer_id', 'RFM_Score'],
    title='RFM Scatter -- Recency vs Monetary (size = Frequency)',
    labels={'recency': 'Recency (days)', 'monetary': 'Total Revenue ($)'},
    opacity=0.65,
    size_max=20
)
fig.update_layout(width=900, height=600)
fig.show()

### 4.5 RFM Heatmap -- Average Monetary by R & F Scores

In [ ]:
heatmap_data = rfm.pivot_table(index='F', columns='R', values='monetary', aggfunc='mean')

fig = px.imshow(
    heatmap_data,
    text_auto='.0f',
    color_continuous_scale='YlOrRd',
    labels={'x': 'Recency Score', 'y': 'Frequency Score', 'color': 'Avg Revenue ($)'},
    title='Average Monetary Value by Recency & Frequency Scores'
)
fig.update_layout(width=700, height=500)
fig.show()

---
## 5. CLTV Modeling (BG-NBD + Gamma-Gamma)

We use probabilistic models from the **lifetimes** library:

- **BG-NBD** models the *number* of future purchases a customer will make.
- **Gamma-Gamma** models the *monetary value* per transaction.
- Combined, they produce a 6-month Customer Lifetime Value prediction.

In [ ]:
# Build the summary table required by lifetimes
cltv_input = summary_data_from_transaction_data(
    transactions,
    customer_id_col='customer_id',
    datetime_col='transaction_date',
    monetary_value_col='total_revenue',
    observation_period_end=reference_date
)

# Filter customers with at least 2 transactions (needed for Gamma-Gamma)
cltv_input = cltv_input[cltv_input['frequency'] > 0]
print(f'Customers eligible for CLTV modeling: {cltv_input.shape[0]:,}')
cltv_input.head()

### 5.1 BG-NBD Model -- Purchase Frequency

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(
    cltv_input['frequency'],
    cltv_input['recency'],
    cltv_input['T']
)
print('BG-NBD Model Parameters:')
print(bgf.summary)

# Predict expected purchases in the next 6 months (180 days)
cltv_input['exp_purchases_6m'] = bgf.predict(
    180,
    cltv_input['frequency'],
    cltv_input['recency'],
    cltv_input['T']
)

print(f'\nAverage expected purchases (6 months): {cltv_input["exp_purchases_6m"].mean():.2f}')

### 5.2 Gamma-Gamma Model -- Monetary Value

In [ ]:
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    cltv_input['frequency'],
    cltv_input['monetary_value']
)
print('Gamma-Gamma Model Parameters:')
print(ggf.summary)

cltv_input['exp_avg_revenue'] = ggf.conditional_expected_average_profit(
    cltv_input['frequency'],
    cltv_input['monetary_value']
)

print(f'\nAverage expected revenue per transaction: ${cltv_input["exp_avg_revenue"].mean():,.2f}')

### 5.3 Six-Month CLTV Prediction

In [ ]:
cltv_input['cltv_6m'] = ggf.customer_lifetime_value(
    bgf,
    cltv_input['frequency'],
    cltv_input['recency'],
    cltv_input['T'],
    cltv_input['monetary_value'],
    time=6,        # months
    freq='D',      # daily frequency in input
    discount_rate=0.01
)

print('CLTV 6-month distribution:')
cltv_input['cltv_6m'].describe()

In [ ]:
# Assign CLTV segments
cltv_input['cltv_segment'] = pd.qcut(
    cltv_input['cltv_6m'], q=4,
    labels=['Low', 'Medium', 'High', 'VIP']
)

cltv_seg = cltv_input['cltv_segment'].value_counts().sort_index()
print('CLTV Segment Counts:')
print(cltv_seg)

In [ ]:
# Top 10 most valuable customers
top10 = (
    cltv_input
    .nlargest(10, 'cltv_6m')
    [['frequency', 'recency', 'T', 'monetary_value',
      'exp_purchases_6m', 'exp_avg_revenue', 'cltv_6m', 'cltv_segment']]
)
top10.style.format({
    'monetary_value': '${:,.0f}',
    'exp_avg_revenue': '${:,.0f}',
    'cltv_6m': '${:,.0f}',
    'exp_purchases_6m': '{:.1f}'
})

In [ ]:
# CLTV distribution by segment
cltv_colors = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#3498db', 'VIP': '#2ecc71'}

fig = px.box(
    cltv_input,
    x='cltv_segment',
    y='cltv_6m',
    color='cltv_segment',
    color_discrete_map=cltv_colors,
    title='6-Month CLTV Distribution by Segment',
    labels={'cltv_6m': 'CLTV ($)', 'cltv_segment': 'Segment'},
    category_orders={'cltv_segment': ['Low', 'Medium', 'High', 'VIP']}
)
fig.update_layout(width=800, height=500, showlegend=False)
fig.show()

In [ ]:
# Expected purchases vs CLTV scatter
fig = px.scatter(
    cltv_input,
    x='exp_purchases_6m',
    y='cltv_6m',
    color='cltv_segment',
    color_discrete_map=cltv_colors,
    opacity=0.5,
    title='Expected Purchases vs 6-Month CLTV',
    labels={'exp_purchases_6m': 'Expected Purchases (6 months)',
            'cltv_6m': 'CLTV ($)'},
    category_orders={'cltv_segment': ['Low', 'Medium', 'High', 'VIP']}
)
fig.update_layout(width=900, height=550)
fig.show()

---
## 6. K-Means Behavioral Clustering

While RFM and CLTV use domain-specific rules and probabilistic models,
K-Means discovers **natural behavioral groups** in the data without prior assumptions.

In [ ]:
# Feature engineering for clustering
cluster_features = (
    transactions
    .groupby('customer_id')
    .agg(
        total_spend      = ('total_revenue', 'sum'),
        avg_spend        = ('total_revenue', 'mean'),
        num_transactions = ('transaction_id', 'nunique'),
        avg_nights       = ('nights', 'mean'),
        avg_daily_rate   = ('daily_rate', 'mean'),
        avg_extra_spend  = ('extra_spend', 'mean')
    )
    .reset_index()
)

print(f'Clustering features for {cluster_features.shape[0]:,} customers')
cluster_features.describe()

In [ ]:
# Standardize features
feature_cols = ['total_spend', 'avg_spend', 'num_transactions',
                'avg_nights', 'avg_daily_rate', 'avg_extra_spend']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_features[feature_cols])
print('Scaled feature matrix shape:', X_scaled.shape)

### 6.1 Elbow Method + Silhouette Score

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f'K={k:2d}  |  Inertia={km.inertia_:>12,.0f}  |  Silhouette={silhouettes[-1]:.4f}')

optimal_k = list(K_range)[np.argmax(silhouettes)]
print(f'\nBest K by Silhouette Score: {optimal_k}')

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Elbow Method (Inertia)', 'Silhouette Score'))

fig.add_trace(
    go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
               marker=dict(size=8, color='#3498db'), name='Inertia'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=list(K_range), y=silhouettes, mode='lines+markers',
               marker=dict(size=8, color='#e74c3c'), name='Silhouette'),
    row=1, col=2
)

# Highlight optimal K
fig.add_vline(x=optimal_k, line_dash='dash', line_color='green', row=1, col=1)
fig.add_vline(x=optimal_k, line_dash='dash', line_color='green', row=1, col=2)

fig.update_xaxes(title_text='Number of Clusters (K)', dtick=1)
fig.update_layout(width=1000, height=400, showlegend=False,
                  title_text=f'Optimal K = {optimal_k}')
fig.show()

### 6.2 Final Clustering & Profiling

In [ ]:
kmeans_final = KMeans(n_clusters=optimal_k, n_init=20, random_state=42)
cluster_features['cluster'] = kmeans_final.fit_predict(X_scaled)

# Human-readable cluster labels will be assigned after profiling
cluster_profile = (
    cluster_features
    .groupby('cluster')[feature_cols]
    .mean()
    .round(2)
)

cluster_sizes = cluster_features['cluster'].value_counts().sort_index()
cluster_profile['count'] = cluster_sizes.values

print('Cluster Profiles (mean values):')
cluster_profile

In [ ]:
# Assign human-readable names based on profile
# Sort clusters by total_spend to create a ranking
spend_rank = cluster_profile['total_spend'].rank(ascending=True).astype(int)

name_map_options = {
    1: 'Budget Travelers',
    2: 'Casual Guests',
    3: 'Regular Guests',
    4: 'Frequent Guests',
    5: 'Premium Guests',
    6: 'VIP Guests',
    7: 'Ultra Premium',
    8: 'Elite',
    9: 'Platinum',
}

cluster_name_map = {cluster_id: name_map_options.get(rank, f'Cluster {rank}')
                    for cluster_id, rank in spend_rank.items()}

cluster_features['cluster_name'] = cluster_features['cluster'].map(cluster_name_map)
cluster_profile.index = cluster_profile.index.map(cluster_name_map)

print('Cluster Name Mapping:', cluster_name_map)
print()
cluster_profile

### 6.3 Radar Chart -- Cluster Comparison

In [ ]:
# Normalize each feature to 0-1 for the radar chart
radar_df = cluster_profile[feature_cols].copy()
for col in feature_cols:
    min_val = radar_df[col].min()
    max_val = radar_df[col].max()
    if max_val > min_val:
        radar_df[col] = (radar_df[col] - min_val) / (max_val - min_val)
    else:
        radar_df[col] = 0.5

radar_colors = px.colors.qualitative.Set2

fig = go.Figure()
for i, (cluster_name, row) in enumerate(radar_df.iterrows()):
    fig.add_trace(go.Scatterpolar(
        r=row.values.tolist() + [row.values[0]],  # close the polygon
        theta=feature_cols + [feature_cols[0]],
        fill='toself',
        name=cluster_name,
        opacity=0.6,
        line=dict(color=radar_colors[i % len(radar_colors)])
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Cluster Radar Profile Comparison',
    width=800,
    height=600
)
fig.show()

In [ ]:
# Cluster size bar chart
fig = px.bar(
    x=list(cluster_name_map.values()),
    y=cluster_sizes.values,
    color=list(cluster_name_map.values()),
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Cluster Size Distribution',
    labels={'x': 'Cluster', 'y': 'Number of Customers'},
    text=cluster_sizes.values
)
fig.update_traces(textposition='outside')
fig.update_layout(width=800, height=450, showlegend=False)
fig.show()

### 6.4 Cluster Scatter (2D PCA Projection)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

cluster_features['pca1'] = X_pca[:, 0]
cluster_features['pca2'] = X_pca[:, 1]

fig = px.scatter(
    cluster_features,
    x='pca1', y='pca2',
    color='cluster_name',
    color_discrete_sequence=px.colors.qualitative.Set2,
    title=f'K-Means Clusters (PCA 2D Projection, explained var = {pca.explained_variance_ratio_.sum():.1%})',
    labels={'pca1': 'PC1', 'pca2': 'PC2', 'cluster_name': 'Cluster'},
    opacity=0.5
)
fig.update_layout(width=900, height=550)
fig.show()

---
## 7. Cross-Analysis: RFM Segments vs Clusters

In [ ]:
# Merge RFM segments with cluster labels
cross = rfm[['customer_id', 'segment']].merge(
    cluster_features[['customer_id', 'cluster_name']],
    on='customer_id'
)

cross_tab = pd.crosstab(cross['segment'], cross['cluster_name'], normalize='index')

fig = px.imshow(
    cross_tab.round(2),
    text_auto='.0%',
    color_continuous_scale='Blues',
    title='RFM Segment vs Behavioral Cluster (Row-Normalized)',
    labels={'x': 'Cluster', 'y': 'RFM Segment', 'color': 'Proportion'}
)
fig.update_layout(width=800, height=500)
fig.show()

---
## 8. Business Recommendations

### RFM-Based Actions

| Segment | Strategy | Example Actions |
|---------|----------|----------------|
| **Champion** | Reward & retain | Exclusive loyalty perks, early access to new properties, personalized concierge |
| **Loyal** | Upsell & engage | Room upgrade offers, spa packages, anniversary recognition |
| **New Customer** | Onboard & convert | Welcome sequence, first-stay discount on return, feedback surveys |
| **Potential** | Nurture | Targeted email campaigns, seasonal promotions, loyalty program invitation |
| **At Risk** | Win back | Re-engagement emails, time-limited offers, satisfaction follow-up |
| **Lost** | Reactivate (selectively) | High-value lost guests get personal outreach; low-value get automated win-back |

### CLTV-Based Prioritization

- **VIP tier** customers drive disproportionate revenue. Assign dedicated relationship managers.
- **High tier** guests are the growth engine. Invest in converting them to VIP through personalization.
- **Medium tier** represents the bulk of the customer base. Optimize operational efficiency for this group.
- **Low tier** requires cost-effective automated engagement. Avoid over-investing in retention.

### Cluster-Driven Personalization

Each behavioral cluster should receive **tailored marketing content**:

- Budget travelers: Value-focused messaging, package deals, off-peak promotions.
- Premium/VIP guests: Luxury experience messaging, exclusive access, premium room categories.
- Extended-stay guests: Long-stay discounts, apartment-style accommodations, loyalty milestones.

---
## 9. Conclusion

This notebook demonstrated three complementary approaches to customer analytics:

1. **RFM segmentation** provides a quick, interpretable view of customer health.
2. **CLTV modeling** (BG-NBD + Gamma-Gamma) quantifies the future dollar value of each guest.
3. **K-Means clustering** discovers natural behavioral groups that cut across RFM boundaries.

**Key takeaways:**

- The Champion and Loyal segments together typically represent around 30% of customers but
  contribute over 60% of total revenue -- protecting these relationships is critical.
- CLTV predictions enable proactive resource allocation: focus human effort on VIP/High guests
  and automate engagement for the long tail.
- Behavioral clusters reveal nuances that RFM alone misses, such as the distinction between
  high-frequency budget travelers and low-frequency premium guests.

**Next steps:**

- Integrate these segments into the hotel CRM for automated campaign triggers.
- Build a real-time scoring API (see `src/models/`) for operational deployment.
- Combine with review sentiment data (NLP pipeline) for a 360-degree guest profile.

---
*Notebook by Mehmet Isik -- Hotel Intelligence Platform*